### 1.卷积神经网络的整体结构

前面学习的神经网络，相邻层的所有神经元都有连接，称之为全连接神经网络，之前实现的是Affine层+ReLU叠加形成最后用softmax输出最终结果(概率)，卷积神经网络与之类似，不过是将Affine-ReLU变为Convolution(卷积层)-ReLU-(Pooling)[池化层]，最后仍然可以使用softmax输出，我们将卷积层的输入输出数据称为特征图，其中输入的叫做输入特征图，输出的叫输出特征图

### 2.为什么要使用卷积神经网络

再次以手写图片识别为例，之前使用的全连接神经网络，是将MNIST中28*28像素的图片压缩到一维，处理简单的图片是可以，但是当处理复杂图形时，维度本身也可以是一种特征，相邻的像素也蕴含着相似的特征，显然武断地将其压缩，不能处理所有问题，于是我们引出卷积神经网络(CNN)

### 3.卷积层

- 基本运算方式

卷积层的运算可以类比Affine层，其中的滤波器相当于权重系数，不过由于卷积神经网络设计的是多维数据，于是滤波器也是一个多维矩阵，它和输入数据进行的是乘积累加计算(这里看图会比较好p205)，之后以固定的步幅移动，输出的同样是一个矩阵，而偏置就是为输出矩阵每一个元素加上偏置值。

-  填充

如果有一个4x4的输入数据，采用3x3的滤波器，输出的应该是一个2x2的矩阵，由此我们看到，如果直接重复卷积层运算，会导致信息矩阵越来越小，最后无法使用，于是我们可以向输入数据的周围填充数据，当我们将4x4矩阵每一个元素周围都填充元素(元素值任意)变成一个6x6矩阵之后，用同样3x3的滤波器，得到的就是4x4的矩阵

- 滤波器的步幅


步幅决定了滤波器的移动间隔，增大步幅会使输出矩阵变小，而填充之后输出矩阵会变大，我们可以定量的用公式计算，若输入大小为(H,W),滤波器大小为(FH,FW),输出大小为(OH,OW),填充为P,步幅为S，则有下述公式：

OH=(H+2P-FH)/S+1

OW=(H+2P-FW)/S+1

上述公式告诉我们输出层和相关因素的关系，要注意，设置数据时必须要使OH，OW为整数，当为小数时，应当采取报错等对策，有时也会四舍五入

- 3维数据的卷积运算

之前学习的是二维数据的卷积运算，即没有厚度，只有行列的二维向量，当出现三维时，也就意味着有多张输入数据，这时自然也需要有多个滤波器对应每一层的输入数据，但是输出数据仍然是二维的，其结果是各个层的输出结果之和

- 结合方块思考


我们知道，当输入数据为三维时(长方体)，我们需要同样三维的滤波器(长方体)，但是最后只能输出二维的矩阵(矩形)，那如何才能使输出结果为一个三维的呢，那就需要为滤波器增加一个维度--个数，即使用多个滤波器，将每个滤波器的结果作为一层，堆叠起来构成一个长方体。当我们为所有数据包括输入数据都加上一个维度(个数)时，就会输出一个四维矩阵，这样就实现了最后一次性输出多个数据，实现了批处理

### 4.池化层

池化是缩小高，长方向上的空间的运算，池化的工具类似于滤波器，池化分为很多种，Max池化，Average池化，Max池化顾名思义，只取出这个范围内最大的数据，Average就是取平均值输出

### 5.卷积层和池化层的实现

1. im2col的使用

因为卷积层和池化层涉及到思维数组，当我们想要访问其中元素时将变得十分困难，所以使用im2col函数，im2col的意思时image to column，从图像到矩阵，非常贴切。它的使用原理是将要被滤波器处理的三维区域转化为适应滤波器的一列(三维转二维)，同时也将三维的滤波器转换为一行，然后使用矩阵相乘，类似于Affine层，就可以实现

2. 卷积层代码实现

首先了解im2col函数的参数：

In [ ]:
#im2col(input_data,filter_h,fliter_w,filter_w,stride=1,pad=0)
#其中input_data是用(数据量，通道，高，长)的四位数组构成的输入数据
#后面依次是滤波器的高，长，步幅，填充

下面试用一下该函数

In [2]:
import numpy as np
import sys,os
sys.path.append(os.pardir)
from common.util import im2col, conv_output_size

x1=np.random.rand(1,3,7,7)#批大小为1，通道为3的7*7数据
col1=im2col(x1,5,5,stride=1,pad=0)
print(col1.shape)
x2=np.random.rand(10,3,7,7)#批大小为10，通道为3的7*7数据
col2=im2col(x2,5,5,stride=1,pad=0)
print(col2.shape)

(9, 75)
(90, 75)


第2维的输出值都是75，是因为这是滤波器的元素的个数总和，而第一维的后一个是前一个的十倍。下面我们仍然用层的方式实现卷积层，便于使用

In [ ]:
class Convolution:
    def __init__(self,w,b,stride=1,pad=0):
        self.w=w
        self.b=b
        self.stride=stride
    def forward(self,x):#注意x是一个四维的数据
        FN,C,FH,FW=self.w.shape
        N,C,H,W=x.shape
        out_h=int(1+(H+2*self.pad-FH)/self.stride)
        out_w=int(1+(W+2*self.pad-FW)/self.stride)
        col=im2col(x,FH,FW,stride=self.stride,pad=0)#这里就是将x用im2col转换
        col_w=self.w.reshape(FN,-1).T#滤波器的展开，这里使用了reshape的便携方法，通过将参数设置为-1，reshape函数会自动计算-1维度上的元素个数，使多维数组的元素个数前后一致，比如(10,3,5,5)形状的数组元素共有750个，指定reshape(10,-1)后，就会转换成(10,75)形状的数组
        out=np.dot(col,col_w)+self.b
        out=out.reshape(N,out_h,out_w,-1).transpose(0,3,1,2)#这一步是将放射变化后的结果变回四维矩阵，但是由于其顺序改变了，所以用transpose函数调整轴的顺序，正确的顺序是(批次，通道，高，宽)，而reshape后的顺序是(批次，宽，通道，高)
        return out

反向传播：

In [ ]:
  def backward(self, dout):
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0,2,3,1).reshape(-1, FN)

        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1, 0).reshape(FN, C, FH, FW)

        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)#与Affine基本一样，除了要使用col2im将其逆处理

        return dx


3. 池化层代码实现

In [ ]:
class Pooling:
    def __init__(self, pool_h, pool_w, stride=2, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad

        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1 + (H - self.pool_h) / self.stride)
        out_w = int(1 + (W - self.pool_w) / self.stride)

        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h*self.pool_w)

        arg_max = np.argmax(col, axis=1)
        out = np.max(col, axis=1)
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

        self.x = x
        self.arg_max = arg_max

        return out

    def backward(self, dout):
        dout = dout.transpose(0, 2, 3, 1)

        pool_size = self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size))
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape + (pool_size,))

        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1)
        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)

        return dx


可以看到其实与卷积层基本一致，除了在通道方向上，池化层是独立的，看图即可(p223)

### 6.CNN的实现

In [4]:
from layers import Convolution, Relu, Pooling, Affine, SoftmaxWithLoss
from collections import OrderedDict
import numpy as np


def numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x)

    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]

        x[idx] = float(tmp_val) + h
        fxh1 = f(x)

        x[idx] = float(tmp_val) - h
        fxh2 = f(x)

        grad[idx] = (fxh1 - fxh2) / (2 * h)

        x[idx] = tmp_val
        it.iternext()

    return grad

class SimpleConvNet:
    def __init__(self, input_dim=(1, 28, 28), conv_param={'filter_num': 30, 'filter_size': 5, 'stride': 1, 'pad': 1}, hidden_size=100, output_size=10, weight_init_std=0.01):
        filter_num = conv_param['filter_num']
        filter_size = conv_param['filter_size']
        filter_stride = conv_param['stride']
        filter_pad = conv_param['pad']
        input_size = input_dim[1]
        conv_output_size = (input_size - filter_size + 2 * filter_pad) / filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size / 2) * (conv_output_size / 2))

        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers['Conv1'] = Convolution(self.params['W1'], self.params['b1'],
                                           conv_param['stride'], conv_param['pad'])
        self.layers['Relu1'] = Relu()
        self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
        self.layers['Relu2'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])

        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    def accuracy(self, x, t, batch_size=100):
        if t.ndim != 1:
            t = np.argmax(t, axis=1)

        acc = 0.0

        for i in range(int(x.shape[0] / batch_size)):
            tx = x[i*batch_size:(i+1)*batch_size]
            tt = t[i*batch_size:(i+1)*batch_size]
            y = self.predict(tx)
            y = np.argmax(y, axis=1)
            acc += np.sum(y == tt)

        return acc / x.shape[0]

    def numerical_gradient(self, x, t):
        loss_w = lambda w: self.loss(x, t)

        grads = {}
        for idx in (1, 2, 3):
            grads['W' + str(idx)] = numerical_gradient(loss_w, self.params['W' + str(idx)])
            grads['b' + str(idx)] = numerical_gradient(loss_w, self.params['b' + str(idx)])

        return grads

    def gradient(self, x, t):
        self.loss(x, t)

        dout = 1
        dout = self.last_layer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        grads = {}
        grads['W1'], grads['b1'] = self.layers['Conv1'].dW, self.layers['Conv1'].db
        grads['W2'], grads['b2'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W3'], grads['b3'] = self.layers['Affine2'].dW, self.layers['Affine2'].db

        return grads


这就是核心层的实现，下面是学习效果：

In [7]:
# coding: utf-8
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

from mnist import load_mnist
from common.trainer import Trainer
from common.layers import Convolution, Relu, Pooling, Affine, SoftmaxWithLoss

# 将数值微分工具函数移到类外面，保持代码整洁
def numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + h
        fxh1 = f(x)
        x[idx] = float(tmp_val) - h
        fxh2 = f(x)
        grad[idx] = (fxh1 - fxh2) / (2 * h)
        x[idx] = tmp_val
        it.iternext()
    return grad


class SimpleConvNet:
    def __init__(self, input_dim=(1, 28, 28), conv_param={'filter_num': 30, 'filter_size': 5, 'stride': 1, 'pad': 0}, hidden_size=100, output_size=10, weight_init_std=0.01):
        filter_num = conv_param['filter_num']
        filter_size = conv_param['filter_size']
        filter_stride = conv_param['stride']
        filter_pad = conv_param['pad']
        input_size = input_dim[1]
        conv_output_size = (input_size - filter_size + 2 * filter_pad) / filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size / 2) * (conv_output_size / 2))

        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers['Conv1'] = Convolution(self.params['W1'], self.params['b1'],
                                           conv_param['stride'], conv_param['pad'])
        self.layers['Relu1'] = Relu()
        self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
        self.layers['Relu2'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])

        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    def accuracy(self, x, t, batch_size=100):
        if t.ndim != 1:
            t = np.argmax(t, axis=1)
        acc = 0.0
        for i in range(int(x.shape[0] / batch_size)):
            tx = x[i*batch_size:(i+1)*batch_size]
            tt = t[i*batch_size:(i+1)*batch_size]
            y = self.predict(tx)
            y = np.argmax(y, axis=1)
            acc += np.sum(y == tt)
        return acc / x.shape[0]

    # 注意：训练时 Trainer 会默认调用这个快速的 gradient 方法
    def gradient(self, x, t):
        self.loss(x, t)
        dout = 1
        dout = self.last_layer.backward(dout)
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Conv1'].dW, self.layers['Conv1'].db
        grads['W2'], grads['b2'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W3'], grads['b3'] = self.layers['Affine2'].dW, self.layers['Affine2'].db
        return grads

    # 这个方法保留着用于学习或调试，但绝不要用于实际的大规模训练！
    def numerical_gradient(self, x, t):
        loss_w = lambda w: self.loss(x, t)
        grads = {}
        for idx in (1, 2, 3):
            grads['W' + str(idx)] = numerical_gradient(loss_w, self.params['W' + str(idx)])
            grads['b' + str(idx)] = numerical_gradient(loss_w, self.params['b' + str(idx)])
        return grads


if __name__ == '__main__':
    (x_train, t_train), (x_test, t_test) = load_mnist(flatten=False)

    max_epochs = 20

    network = SimpleConvNet(input_dim=(1,28,28),
                            conv_param = {'filter_num': 30, 'filter_size': 5, 'pad': 0, 'stride': 1},
                            hidden_size=100, output_size=10, weight_init_std=0.01)

    trainer = Trainer(network, x_train, t_train, x_test, t_test,
                      epochs=max_epochs, mini_batch_size=100,
                      optimizer='Adam', optimizer_param={'lr': 0.001},
                      evaluate_sample_num_per_epoch=1000)
    trainer.train()

    network.save_params("params.pkl")
    print("Saved Network Parameters!")

    markers = {'train': 'o', 'test': 's'}
    x = np.arange(max_epochs)
    plt.plot(x, trainer.train_acc_list, marker='o', label='train', markevery=2)
    plt.plot(x, trainer.test_acc_list, marker='s', label='test', markevery=2)
    plt.xlabel("epochs")
    plt.ylabel("accuracy")
    plt.ylim(0, 1.0)
    plt.legend(loc='lower right')
    plt.show()

train loss:2.299336138580474
=== epoch:1, train acc:0.234, test acc:0.227 ===
train loss:2.294719481134759
train loss:2.292817947293037
train loss:2.282280504702465
train loss:2.2729846832390916
train loss:2.2639551613275946
train loss:2.241603629244575
train loss:2.2103983929215376
train loss:2.2206593797886423
train loss:2.1883299367789975
train loss:2.152263348934689
train loss:2.1182638791951387
train loss:2.0558288694196056
train loss:2.087055342931755
train loss:1.959520343683283
train loss:1.8773145250698133
train loss:1.9228920476592775
train loss:1.9212569548436775
train loss:1.7699746389455322
train loss:1.7262089747573952
train loss:1.6312443495614009
train loss:1.5521925521241378
train loss:1.3986347646626056
train loss:1.4112363232350174
train loss:1.3756757036808511
train loss:1.1866413698983451
train loss:1.1649893477778894
train loss:1.0992519761890696
train loss:1.0078483872373858
train loss:1.1024747188753388
train loss:0.9685623282659103
train loss:0.8677771803751054

KeyboardInterrupt: 